# Sistema de Asistencia Inteligente - Unimarc
## Gestión de Inventario y Operaciones

In [ ]:
import os
import re
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# #1. CARGAR CONFIGURACIÓN DESDE EL .ENV
load_dotenv()

MODEL_ID = "gpt-4.1"

# #2. CONFIGURACIÓN DE LOS MODELOS (LLM)
# Se configuran usando las llaves exactas de tu archivo .env

# LLM para la conversación (con Streaming)
llm = ChatOpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL"),
    api_key=os.environ.get("GITHUB_TOKEN"),
    model=MODEL_ID,
    temperature=0.3,
    streaming=True,
    max_tokens=300
)

# LLM para procesos de extracción y resumen (Determinista)
llm_util = ChatOpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL"),
    api_key=os.environ.get("GITHUB_TOKEN"),
    model=MODEL_ID,
    temperature=0,
    streaming=False
)


In [19]:
# #3. ESTRUCTURA PARA EL HISTORIAL
sesion_unimarc = {}

def historial_de_conversacion(sesion_id : str):
    if sesion_id not in sesion_unimarc:
        sesion_unimarc[sesion_id] = InMemoryChatMessageHistory()
    return sesion_unimarc[sesion_id]

# #4. FUNCIÓN PARA EXTRAER INFORMACIÓN DE INVENTARIO (Texto original restaurado)
def extraer_datos_inventario(texto_usuario: str) -> dict:
    """
    Analiza el texto del usuario y extrae un JSON estructurado de productos.
    """
    prompt_extraccion = (
        "Eres un experto en logística de Unimarc. "
        "Extrae los datos de inventario del siguiente texto y entrégalos "
        "EXCLUSIVAMENTE en formato JSON. No incluyas saludos ni explicaciones.\n\n"
        "Si el texto NO contiene información de inventario, responde solo: null\n\n"
        "Formato requerido:\n"
        "{\n"
        "  \"productos\": [\n"
        "    {\"nombre\": \"nombre del producto\", \"cantidad\": numero, \"unidad\": \"kg/unidades/cajas\"}\n"
        "  ]\n"
        "}\n\n"
        f"Texto: \"{texto_usuario}\"\n"
        "JSON:"
    )
    
    try:
        response = llm_util.invoke([HumanMessage(content=prompt_extraccion)])
        contenido = response.content.strip()
        
        # Limpieza de bloques de código markdown
        contenido = re.sub(r'```json|```', '', contenido).strip()

        # Búsqueda del bloque entre llaves
        match = re.search(r'\{.*\}', contenido, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        return None
    except Exception as e:
        print(f"[DEBUG ERROR EXTRAER]: {e}")
        return None

# #5. FUNCIÓN PARA SINCRONIZAR CONTEXTO Y RESUMIR
def sincronizar_contexto_stock(sesion_id: str, max_mensajes=6):
    historial = historial_de_conversacion(sesion_id)

    if len(historial.messages) > max_mensajes:
        mensajes_a_resumir = historial.messages[:-2]
        recientes = historial.messages[-2:]

        conversation_text = ""
        for msj in mensajes_a_resumir:
            role = "Usuario" if msj.type == "human" else "Asistente"
            conversation_text += f"{role}: {msj.content}\n"
        
        prompt_resumen = (
            "Eres un experto en logística de Unimarc. Resume la siguiente conversación. "
            "REGLA DE ORO: No omitas códigos de productos, nombres de artículos ni cantidades de stock. "
            "Resume el resto en máximo 2 líneas de forma técnica.\n\n"
            f"Historial a resumir:\n{conversation_text}"
        )

        summary_response = llm_util.invoke(prompt_resumen)
        summary = summary_response.content
        
        historial.clear()
        historial.add_ai_message(f"[MEMORIA DE STOCK]: {summary}")
        historial.messages.extend(recientes)

# #6. PROMPT DE CONVERSACIÓN ORIGINAL RESTAURADO
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente de Inventario de Unimarc. \n"
    "Ayudas a los usuarios a gestionar su inventario, responder preguntas sobre productos, y proporcionar información relevante sobre el stock y las operaciones de la tienda.\n" 
    "Tienes que ser amigable, eficiente y siempre estar dispuesto a ayudar. \n"
    "Si no sabes la respuesta a una pregunta, es mejor admitirlo que inventar una respuesta incorrecta.\n"
    "Siempre debes mantener un tono profesional y cortés en tus respuestas. \n"
    "Las respuestas deben ser breves y al punto, evitando información innecesaria. \n"
    "Si el usuario hace una pregunta que no está relacionada con el inventario o las operaciones de la tienda, debes redirigir la conversación de vuelta a temas relevantes para Unimarc.\n"
    "Recuerda que tu objetivo principal es ayudar a los usuarios a gestionar su inventario de manera efectiva y proporcionar información precisa sobre los productos y operaciones de la tienda.\n"
    "No uses emojis, manten limpio el formato de tus respuestas.\n"
    "Escribe en bloques de texto ANGOSTOS.\n"
    "Presiona 'Enter' (salto de línea) cada 8 o 10 palabras.\n"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

conversation = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history=historial_de_conversacion,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# #7. FUNCIÓN DE EJECUCIÓN
def ejecutar_chat(input_text, session_id):
    sincronizar_contexto_stock(session_id)
    
    datos = extraer_datos_inventario(input_text)
    if datos:
        print(f"\n[SISTEMA - LOG]: Datos extraídos: {datos}")

    config = {"configurable": {"session_id": session_id}}
    
    print(f"\n[USUARIO]: {input_text}")
    print(f"[OUTPUT]: ", end="", flush=True)
    
    try:
        for chunk in conversation.stream({"input": input_text}, config=config):
            if chunk.content:
                print(chunk.content, end="", flush=True)
        print("\n")
    except Exception as e:
        print(f"\n[ERROR_LOG]: {e}")

# #8. BUCLE TERMINAL
if __name__ == "__main__":
    SID = "SYS-LOG-01"
    print(f"--- TERMINAL UNIMARC | MODELO: {MODEL_ID} ---")
    while True:
        user_input = input("[INPUT]: ")
        if user_input.lower() in ["salir", "exit", "quit"]:
            break
        if user_input.strip():
            ejecutar_chat(user_input, SID)

--- TERMINAL UNIMARC | MODELO: gpt-4.1 ---



[SISTEMA - LOG]: Datos extraídos: {'productos': [{'nombre': 'vino', 'cantidad': 50, 'unidad': 'cajas'}, {'nombre': 'vino', 'cantidad': 250, 'unidad': 'unidades'}, {'nombre': 'aceite', 'cantidad': 20, 'unidad': 'cajas'}, {'nombre': 'aceite', 'cantidad': 240, 'unidad': 'unidades'}, {'nombre': 'galletas mckay', 'cantidad': 5, 'unidad': 'cajas'}, {'nombre': 'galletas mckay', 'cantidad': 100, 'unidad': 'unidades'}]}

[USUARIO]: Han llegado 50 cajas de vino de 5 unidades cada una, 20 cajas de aceite de 12 unidades cada una y 5 cajas de galletas mckay con 20 galletas cada una
[OUTPUT]: Registro la llegada de los productos al inventario:

- Vino: 50 cajas x 5 unidades = 250 unidades.
- Aceite: 20 cajas x 12 unidades = 240 unidades.
- Galletas McKay: 5 cajas x 20 unidades = 100 unidades.

¿Desea que actualice el inventario con estos
nuevos ingresos o necesita información adicional
sobre alguno de estos productos?


[SISTEMA - LOG]: Datos extraídos: {'productos': [{'nombre': 'vino', 'cantidad':